# Topic Modeling con LDA

Proyecto de modelado de temas listo para **Google Colab**.

**Pregunta:** ¿cuáles son los temas principales en un conjunto de documentos?

**Método:** Latent Dirichlet Allocation (LDA)

**Stack:** pandas, scikit-learn, matplotlib

Documentos simulados de ejemplo.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

RANDOM_STATE = 42
print("OK")

## 1. Generar documentos de ejemplo

In [ ]:
np.random.seed(RANDOM_STATE)

# Temas simulados
tema_deporte = ["equipo gana partido campeonato jugador gol", "partido fútbol victoria derrota", "campeonato liga jugador estrella"]
tema_tecnologia = ["nuevo software aplicación inteligencia artificial", "computadora programa código desarrollo", "tecnología innovación digital"]
tema_comida = ["restaurante comida deliciosa receta cocina", "plato sabor ingredientes restaurante", "cocina chef menú especial"]

documentos = []
for _ in range(100):
    tema = np.random.choice([tema_deporte, tema_tecnologia, tema_comida])
    doc = " ".join(np.random.choice(tema, size=3))
    documentos.append(doc)

df = pd.DataFrame({"documento" : documentos})
print(df.shape)
df.head()

## 2. Vectorización

In [ ]:
vectorizer = CountVectorizer(max_features=100, stop_words="spanish")
X = vectorizer.fit_transform(df["documento"])
print(X.shape)
print("Vocabulario:", len(vectorizer.vocabulary_))

## 3. LDA

In [ ]:
n_topics = 3
lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_STATE)
lda.fit(X)
print("LDA entrenado")

## 4. Mostrar temas

In [ ]:
def mostrar_temas(model, feature_names, n_top_words=5):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Tema {topic_idx}: {', '.join(top_words)}")

mostrar_temas(lda, vectorizer.get_feature_names_out())

## 5. Distribución de temas por documento

In [ ]:
topic_dist = lda.transform(X)
df_topics = pd.DataFrame(topic_dist, columns=[f"Tema_{i}" for i in range(n_topics)])
df_topics["tema_dominante"] = df_topics.idxmax(axis=1)
print(df_topics["tema_dominante"].value_counts())
df_topics.head()

In [ ]:
plt.figure(figsize=(6, 4))
df_topics["tema_dominante"].value_counts().plot(kind="bar")
plt.title("Distribución de temas dominantes")
plt.xticks(rotation=0)
plt.show()

## 6. Conclusiones

- LDA recupera los temas simulados (deporte, tecnología, comida).
- Útil para explorar grandes colecciones de texto.
- Se puede extender a noticias reales, reviews, tweets, etc.

**Consejo:** ajustar `n_topics` y preprocesar el texto (stopwords, stemming).